# CDOT TDM VMT Calculator

## Strategy showcase

A walk through the production-ready VMT-reduction strategies, the data behind them, and how they look on real Colorado TAZs.


## Data sources behind every number

Everything in this calculator traces back to one of these sources — no hand-tuned numbers.

| Layer | Source | Granularity | Used for |
|---|---|---|---|
| TAZ stocks & VMT | **CDOT 2019 TDM** (`CDOT_2019_TAZ.json`) | 8,045 TAZ polygons | population, employment, daily VMT, trips |
| Road network | **CDOT 2019 TDM** + **CDOT Open Data** (Highways, Major/Local Roads, PACE) | 84K state-hwy segments, 224K minor-road segments, 9,340 PACE segments | lane-miles, AADT, LTS, observed bike ridership |
| Transit service | **CDOT 2019 TDM** + **CDOT Statewide Transit GTFS** | per-TAZ PMT/VRH + 13K stops, 1K routes | transit service intensity |
| Commute mode share | **ACS B08301 2022 5-Year** | block group (~3,500 in CO) | observed transit / auto / bike / walk shares |
| Bikeable days | **NOAA NCEI 1991-2020 Daily Climate Normals** | 30 CO HCN/CRN/GSN stations + IDW interpolation | per-TAZ cycling-comfort days |
| Urban / rural | **CDOT Urban Areas 2020** + density-based fallback | per-TAZ polygon overlay | area-type classification |
| Elasticities & effect sizes | **CAPCOA Handbook 2021**, **CARB 2025 Policy Briefs**, peer-reviewed studies | per parameter | strategy-by-strategy (cited on each slide) |

Behavioral defaults (AVO, parking prices, vehicle ownership cost) come from NHTS 2017 / AAA — flagged on each strategy where used.

## Data pipeline

`prepare_taz()` produces a single 8,045-row × ~95-column table merging all the above:

```
  CDOT TDM TAZ  +  CDOT external (cached)  +  ACS block-group  +  NOAA NCEI
        │                  │                       │                 │
        └──────────────────┴───────────────────────┴─────────────────┘
                                    │
                          prepare_taz.prepare_taz()
                                    │
                         per-TAZ DataFrame  ─►  strategy_*()  ─►  VMT-reduction table
```

**Priority rule for behavioral inputs:** observed per-TAZ value → area-type default. Every strategy result carries a `data_assumptions` column listing which defaults were applied.

In [ ]:
# Setup (hidden in presentation)
import sys, pandas as pd
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from prepare_taz import prepare_taz
import strategy_calculations as sc

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 20)

_prep = prepare_taz()
_taz  = sc.add_imputed_parking(sc.add_imputed_avo(sc.add_imputed_mode_shares(_prep.df)))

# Demo TAZ set: one per area type, picked deterministically.
demo = pd.concat([
    _taz[_taz['area_type'] == 'urban_core'].nlargest(1, 'activity_density'),
    _taz[(_taz['area_type'] == 'urban') & (_taz['population'] > 1000)].nlargest(1, 'population'),
    _taz[(_taz['area_type'] == 'suburban') & (_taz['employment'] > 1000)].nlargest(1, 'employment'),
    _taz[_taz['area_type'] == 'rural'].nlargest(1, 'daily_vmt'),
]).reset_index(drop=True)

def fmt(res):
    """Plain-DataFrame view of a strategy result for the 4 demo TAZs."""
    out = res.merge(demo[['taz_id','county','area_type']], on='taz_id')
    out = out[['taz_id','county','area_type','base_vmt','pct_vmt_reduction','daily_vmt_reduction']].copy()
    out['base_vmt']            = out['base_vmt'].round(0).astype(int)
    out['pct_vmt_reduction']   = (out['pct_vmt_reduction'] * 100).round(2).map(lambda v: f'{v:+.2f}%')
    out['daily_vmt_reduction'] = out['daily_vmt_reduction'].round(0).astype(int)
    return out

## Demo TAZs

One representative TAZ from each area type — same four used for every example below.

In [ ]:
demo[['taz_id','county','district','area_type','population','employment','daily_vmt']].assign(
    population=lambda d: d['population'].round(0).astype(int),
    employment=lambda d: d['employment'].round(0).astype(int),
    daily_vmt=lambda d: d['daily_vmt'].round(0).astype(int),
)

## Density Change

Increases in residential and/or employment density reduce per-capita VMT through better trip-chaining, mode choice, and shorter trips. The formula multiplies residential and employment effects so combined gains compound.

**Formula**

$$\%\Delta\text{VMT} = (1 + r_\text{res})(1 + r_\text{emp}) - 1$$
$$r_\text{res} = \%\Delta\rho_\text{res} \cdot \varepsilon_\text{res}, \quad r_\text{emp} = \%\Delta\rho_\text{emp} \cdot \varepsilon_\text{emp}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `pct_change_res_density` | +20% | Residential density change |
| `pct_change_emp_density` | +20% (optional) | Employment density change |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Residential density elasticity | -0.22 | Stevens 2016 (JAPA 83) — meta-analysis controlling for self-selection |
| Employment density elasticity | -0.07 | Stevens 2016 |

In [ ]:
# +20% residential + +20% employment density (mixed-use intensification)
fmt(sc.strategy_density_change(demo, pct_change_res_density=0.5, pct_change_emp_density=0.20))

## Separated & Protected Bike Lanes

Adding a separated facility on a corridor captures some share of parallel-road VMT as bike trips, scaled by the climate's bikeability and trip-length ratios.

**Formula**

$$\%\Delta\text{VMT} = -\, p_\text{parallel} \cdot \frac{\text{bikeable days}}{365} \cdot \varepsilon \cdot \frac{L_\text{bike}}{L_\text{veh}}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `pct_parallel_vmt_affected` | 5% | Share of TAZ VMT on the facility's parallel roads |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Bike-facility effect size (ε) | +0.07 | CAPCOA T-21 effect size midpoint |
| Annual bikeable days | **per-TAZ** | NOAA NCEI 1991-2020 daily normals (TMAX in 32-95°F, weighted by precip probability), IDW-interpolated from 30 CO stations |
| Avg bike trip length | 1.5 mi | NACTO Shared Micromobility avg |
| Avg vehicle trip length | **per-TAZ** | TDM `VMT / rptTrips` |

In [ ]:
fmt(sc.strategy_separated_bike_lanes(demo, pct_parallel_vmt_affected=0.15))

## Bike Mode-Share Booster (Sharrows + End-of-Trip Facilities)

Lower-cost bike interventions that boost the existing bike mode share by a small adjustment factor. The `scope` switch picks between area-wide effects (sharrows on a road network) and commute-specific effects (workplace end-of-trip facilities like showers + secure parking).

**Formula**

$$\%\Delta\text{VMT} = -\, s_\text{scope} \cdot \frac{L_\text{bike} \cdot \text{MS}_\text{bike} \cdot \text{adj}}{L_\text{veh} \cdot \text{MS}_\text{auto}}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `scope` | `'area_vmt'` or `'commute'` | Sharrows (area) vs. End-of-trip (commute) |
| `scope_share` | 10% area / 40% employees | Fraction of trips affected |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Sharrows mode-share boost | +15% | CAPCOA T-19 midpoint |
| End-of-trip mode-share boost | +5% | CAPCOA T-29 midpoint |
| Bike / auto mode share | **per-TAZ** | ACS B08301 commute, or area-type fallback |
| Avg bike trip length | 1.5 mi | NACTO |

In [ ]:
sharrows = fmt(sc.strategy_bike_mode_share_booster(demo, scope_share=0.10, scope='area_vmt')).assign(variant='sharrows (area)')
eot      = fmt(sc.strategy_bike_mode_share_booster(demo, scope_share=0.40, scope='commute')).assign(variant='end-of-trip (commute)')
pd.concat([sharrows, eot])[['variant','taz_id','county','area_type','base_vmt','pct_vmt_reduction','daily_vmt_reduction']]

## Transit Service Expansion

Either more frequent service on existing routes (lower wait times → more riders) or new routes / extended hours (more service miles → more access). The `basis` switch picks the elasticity.

**Formula**

$$\%\Delta\text{VMT} = -L \cdot \frac{\Delta\text{service} \cdot \text{MS}_\text{transit} \cdot \varepsilon}{\text{AVO} \cdot \text{MS}_\text{auto}}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `basis` | `'frequency'` or `'service_miles'` | Picks the elasticity |
| `pct_change` | +25% freq, +20% miles | Signed change in service |
| `level_of_implementation` | 60% | Share of TAZ's routes affected |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Frequency elasticity | +0.50 | Handy 2013 (CARB Tech Background) |
| Service-miles elasticity | +0.75 | TCRP Report 95 Ch. 9 midpoint of 0.7–0.8 |
| Trip-reduction ratio (service-miles) | 0.7 | TCRP 95 |
| Transit / auto mode share | **per-TAZ** | ACS B08301 or area-type |
| AVO (auto occupancy) | 1.20 | NHTS 2017 national avg |

In [ ]:
fmt(sc.strategy_transit_service_expansion(demo, pct_change=0.25, basis='frequency',
                                          level_of_implementation=0.60))

## Shared Micromobility

Deploying shared bikes, e-bikes, or scooters substitutes some share of auto trips. Different vehicle types have different car-substitution rates.

**Formula**

$$\%\Delta\text{VMT} = -\, \Delta\text{access} \cdot \frac{T_\text{micro} \cdot r_\text{sub} \cdot L_\text{micro}}{T_\text{veh} \cdot L_\text{veh}}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `micromobility_type` | `bikeshare` / `e-bikeshare` / `scootershare` | Picks the substitution ratio |
| `pct_pop_access_before` | 0% | Baseline share of pop with access |
| `pct_pop_access_after` | 30% | Share with access after deployment |

**Tunable defaults (per type)**

| Type | Substitution ratio | Source |
|---|---|---|
| `bikeshare` | 19.6% | McQueen et al. 2020 |
| `e-bikeshare` | 35.0% | Fitch et al. 2021 |
| `scootershare` | 38.5% | McQueen et al. 2020 |
| Daily micro trips / person | 0.05 | NACTO Shared Micromobility State of Practice |
| Avg micro trip length | 1.0 mi | NACTO |

In [ ]:
rows = []
for mtype in ['bikeshare','e-bikeshare','scootershare']:
    r = sc.strategy_shared_micromobility(demo, pct_pop_access_before=0.0,
                                          pct_pop_access_after=0.30, micromobility_type=mtype)
    rows.append(fmt(r).assign(type=mtype))
pd.concat(rows)[['type','taz_id','county','area_type','base_vmt','pct_vmt_reduction','daily_vmt_reduction']]

## Transit Oriented Development

Residents within the TOD walkshed have transit mode share ~4.9× the surrounding area, capped at a realistic ceiling. Scaled to the TAZ by the share of TAZ population that lives in the TOD.

**Formula**

$$\%\Delta\text{VMT} = -\, p_\text{TOD} \cdot \frac{\text{MS}_\text{TOD} - \text{MS}_\text{transit}}{\text{MS}_\text{auto}}$$
$$\text{MS}_\text{TOD} = \min(\text{MS}_\text{transit} \cdot 4.9,\; 0.50)$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `pct_taz_in_tod` | 10% (small) – 30% (heavy) | Share of TAZ population within the TOD walkshed |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| TOD transit mode share ratio | **4.9×** | CAPCOA LUT-4 (citing Lund 2004, Cervero 2007, Chatman 2013) |
| Max TOD transit mode share | 50% | Realistic ceiling even for rail-station TOD |
| Transit / auto mode share | **per-TAZ** | ACS B08301 or area-type |

In [ ]:
fmt(sc.strategy_transit_oriented_development(demo, pct_taz_in_tod=0.10))

## Vanpool

Employer-supported vanpool program treated as a small-scale transit-style expansion scoped to commute VMT. Most effective on long suburban-to-employment corridors.

**Formula**

$$\%\Delta\text{Commute VMT} = -\, p_\text{trips} \cdot \Delta\text{service} \cdot \frac{\text{MS}_\text{transit} \cdot \varepsilon}{\text{AVO}}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `pct_trips_impacted` | 5% | Share of commute trips a vanpool program can reach |
| `pct_service_change` | +100% (default) | Service change vs. baseline |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Service elasticity | +0.75 | TCRP 95 service-hours elasticity midpoint |
| Transit mode share | **per-TAZ** | ACS B08301 or area-type |
| AVO | 1.20 | NHTS 2017 |

In [ ]:
fmt(sc.strategy_vanpool(demo, pct_trips_impacted=0.05))

## TMO Coverage

Growing the share of employees covered by a Transportation Management Organization that runs a voluntary commute-trip-reduction program.

**Formula**

$$\%\Delta\text{Commute VMT} = -(s_\text{after} - s_\text{before}) \cdot r_\text{CTR}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `share_before` | 0% | Existing TMO coverage |
| `share_after` | 40% | Target coverage |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Per-eligible commute-VMT reduction | 4% | CAPCOA TRT-1 voluntary CTR midpoint (range 2-8%) |
| Commute VMT share | 30% | NHTS 2017 work-trip share |

In [ ]:
fmt(sc.strategy_tmo_coverage(demo, share_before=0.0, share_after=0.40))

## Commute Program (Marketing / Incentives)

Soft commute-program interventions like outreach campaigns, financial rewards for alt-mode use, employer trip-reduction marketing.

**Formula**

$$\%\Delta\text{Commute VMT} = -\, p_\text{eligible} \cdot r_\text{per-eligible}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `pct_eligible` | 50% | Share of employees reached |
| `reduction_per_eligible` | 2% (default) | Tune for marketing (1%) vs incentives (3%) |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Marketing campaign effect size | 1% | CAPCOA T-9 |
| Incentive campaign effect size | 3% | CAPCOA T-13 |
| Default midpoint | 2% | Conservative single-program estimate |

In [ ]:
fmt(sc.strategy_commute_program(demo, pct_eligible=0.50))

## Telework

Eliminates commute round-trips on telework days. Most defensible strategy in the calculator — the formula is purely mechanical (no contested elasticity).

**Formula**

$$\%\Delta\text{Commute VMT} = -\, p_\text{eligible} \cdot \frac{\text{days/wk}}{5}$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `pct_eligible` | 50% | Share of workforce with telework option |
| `telework_days_per_week` | 2 | Average days teleworked per eligible employee |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Net-VMT factor | 1.0 (full credit) | Mechanical accounting. CARB Telecommuting 2025 brief suggests 0.75–0.85 factor for induced non-commute trips — could refine. |
| Commute VMT share | 30% | NHTS 2017 work-trip share |

In [ ]:
fmt(sc.strategy_telework(demo, pct_eligible=0.50, telework_days_per_week=2))

## Lane-Mile Addition (Induced Demand)

**Adding capacity increases VMT.** Long-run induced-demand elasticity is the most-replicated finding in transportation economics — added freeway lane-miles induce roughly proportional new VMT.

**Formula**

$$\%\Delta\text{VMT} = \frac{\Delta\text{lane-miles}}{\text{existing lane-miles}} \cdot \varepsilon$$


**User inputs**

| Input | Example | Meaning |
|---|---|---|
| `new_lane_miles` | 2.0 | Added through-lane-miles |
| `facility_class` | `'freeway'`, `'major_arterial'`, etc. | Picks the elasticity |

**Tunable defaults**

| Default | Value | Source |
|---|---|---|
| Freeway elasticity | **+1.0** | Duranton & Turner 2011 (AER) long-run |
| Arterial elasticity | +0.6 | D&T 2011 / Hymel 2019 |
| Collector elasticity | +0.4 | D&T 2011 / Hymel 2019 |
| Existing lane-miles in class | **per-TAZ** | CDOT loaded network spatial join |

Note: positive `pct_vmt_reduction` = VMT increase; negative `daily_vmt_reduction` = miles added.

In [ ]:
# Add 2 lane-miles of major arterial to TAZs that already have some
demo_w_arterial = demo[demo['lane_mi_major_arterial'] > 1].copy()
fmt(sc.strategy_lane_mile_addition(demo_w_arterial, new_lane_miles=2.0, facility_class='major_arterial'))

## Where this goes from here

**Production-ready (shown today): 11 of 19 strategies**

**Also available, needs more review:**
- Transit Fare Subsidy — depends on per-zone fare lookup (GTFS extraction queued)
- Parking Pricing / Unbundled Parking / Parking Cash-Out — per-TAZ parking prices have no public source; area-type defaults still in use
- Affordable Housing / Infill — single-source CAPCOA effect size

**Per Handy et al. 2025 (CARB 2025 briefs), quantification not recommended; included as effect-size estimates:**
- Park and Ride · Mobility Hub · Traffic Calming

**Stacking guidance** — multiple strategies on the same TAZ combine multiplicatively (`retained_vmt = ∏(1 + rᵢ)`); some pairs target the same decision and must not be stacked (e.g. Parking Pricing commute + Parking Cash-Out).

**Open follow-ups** — GTFS service-frequency aggregation; ACS vehicle availability (B25044) for sharper Unbundled Parking targeting; OSM bike-network for local-street coverage beyond PACE.